In [1]:
import numpy as np
from astropy.nddata import block_replicate, block_reduce

In [8]:
def upscale(
    data: np.array,
    upscale_y: int = 1,
    upscale_x: int = 1,
) -> np.array:
    """
    Upscaling.
    """
    if not (
        (isinstance(upscale_y, int) and upscale_y > 0) and
        (isinstance(upscale_x, int) and upscale_x > 0)
    ):
        raise ValueError("Upscaling factors must be positive integers.")
    
    for i, f in enumerate((upscale_y, upscale_x)):
        data = np.repeat(data, f, axis=i)
    
    return data/np.prod((upscale_y, upscale_x))


def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale.
    """
    def _handle_shape(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Adjusts input array to be subdivided into blocks."""
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = data[:adj_shape[ax]]
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))

In [9]:
import mbloodmoon as bm
from IROS_pipeline import _handle_dirpaths

mask_FITS = "wfm_mask.fits"

skyfield = "GalacticCenter"
data_FITS = "20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb"

mask_file, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
)

wfm = bm.codedmask(mask_file, upscale_x=1, upscale_y=1)

In [17]:
sky = np.ones(wfm.sky_shape)

up_sky = upscale(sky, *(5, 3))
down_sky = downscale(up_sky, *(5, 3))


sky.shape, up_sky.shape, down_sky.shape, sky.sum(), up_sky.sum(), down_sky.sum()

((1033, 1671),
 (5165, 5013),
 (1033, 1671),
 np.float64(1726143.0),
 np.float64(1726142.9999999441),
 np.float64(1726143.0))

In [15]:
wfm2 = bm.codedmask(mask_file, upscale_x=3, upscale_y=5)

wfm2.sky_shape

(5163, 5015)

In [ ]:
def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale.
    """
    def _handle_shape(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Adjusts input array to be subdivided into blocks."""
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = data[:adj_shape[ax]]
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))



a = np.ones((12, 15))
fy, fx = 5, 5
reduced_a = block_reduce(a, (fy, fx))
downsampled_a = downscale(a, *(fy, fx))


print(
    np.all(downsampled_a == reduced_a),
    a.shape,
    reduced_a.shape,
    downsampled_a.shape,
    a,
    downsampled_a,
)

True (12, 15) (2, 3) (2, 3) [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]] [[25. 25. 25.]
 [25. 25. 25.]]


In [ ]:
np.sum(a), np.sum(downsampled_a), np.sum(reduced_a)

(np.float64(180.0), np.float64(150.0), np.float64(150.0))